# Prompt Engineering

1. In-Context Learning
    It allows the model to carry out a new task in inference, while taking advantage of the knowledge it was trained with by mapping a new relationship.
    The paper *Language Models are Few Shot Learners* the authors describe LLM as few shot learners as a set of example in the prompt is given and the model can map relationship between input and output and have learned a new task. But the model is actually not learning anything new but rather exploiting the relationship that have been learnt in pre-training.

**Advantages of context learning**
- Mirrors the human cognitive reasoning process, so it makes it easier to describe a problem and exploit an LLM.
- It does not require parameter upgrade, so it is fast and only requires few example.
- Model can achieve competitive performance in several benchmarks.

In [1]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 176.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2


In [2]:
!pip install huggingface_hub

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from dotenv import load_dotenv, dotenv_values
import os

In [ ]:
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

In [5]:
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [8]:
input_text = "Hello how are you?"
input_tokens = tokenizer(input_text, return_tensors="pt")


In [9]:
output = model.generate(**input_tokens)
output_text  = tokenizer.decode(output[0], skip_special_tokens=True)

output_text

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"Hello how are you?\n\nI'm doing well, thanks for asking! How about you?"

**Zero shot Prompting**\
The example below is of zero-shot prompting because it neither contains examples nor demonstrations. The model correctly answers just a simple question "When was " As this model from Mistral successfully responds to the question, it can be said that "Mistral 7B has zero shot capabilities"

In [10]:
input_question = "When was Einstein born?"
input_q_one_tokens = tokenizer(input_question, return_tensors = "pt")

In [11]:
output_ans_1 = model.generate(**input_q_one_tokens)
output_ans_1_text = tokenizer.decode(output_ans_1[0], skip_special_tokens=True)

output_ans_1_text

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'When was Einstein born?\nAlbert Einstein was born on March 14, 1879.'

**Few shot Prompting**\
Now we perform few shot prompting where we give some examples along with the solution or required answer type.

In [12]:
few_shot_prompts = """[INST]
Sentence: The food was fantastic, I absolutely loved the taste of turmeric.| Sentiment: Positive,
Sentence: Oh! I am sorry I could not feel any taste, the steak was uncooked and the soup was horrible | Sentiment: Negative,
Sentence: Absolutely delicious, the ice-cream, it felt like I am sitting on the lap of Himalayas and enjoying the most delicious taste of real strawberry. | Sentiment: Positive
Sentence: I don't know,the meat did not feel tender and still looked bright red. Also the ice cream, just looks pink and does not have any flavor.| Sentiment:
[/INST]"""

In [13]:
input_fs_prompt_tokens = tokenizer(few_shot_prompts, return_tensors="pt")

In [14]:
output_fs_prompt = model.generate(**input_fs_prompt_tokens)
output_fs_prompt_text = tokenizer.decode(output_fs_prompt[0], skip_special_tokens=True)

output_fs_prompt_text

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"[INST]\nSentence: The food was fantastic, I absolutely loved the taste of turmeric.| Sentiment: Positive,\nSentence: Oh! I am sorry I could not feel any taste, the steak was uncooked and the soup was horrible | Sentiment: Negative, \nSentence: Absolutely delicious, the ice-cream, it felt like I am sitting on the lap of Himalayas and enjoying the most delicious taste of real strawberry. | Sentiment: Positive\nSentence: I don't know,the meat did not feel tender and still looked bright red. Also the ice cream, just looks pink and does not have any flavor.| Sentiment: \n[/INST] Negative."

**Chain of Thought**\
The simple prompting has some limitations, especially when it comes to tasks that requires reasoning. Sometimes giving examples are quite not enough to guide the model to a right direction and hence various techniques are proposed to avoid fine tuning. In **Chain of Thought (COT)** approach, a triplet <input, chain of thought, output> is provided as prompt as in the example below.

In [15]:
cot_prompt = """[INST]
Look at the following problem and how to solve it step by step.
Alexa has 5 pencils. She asks her father to bring 3 more and then gives 2 to her friend. How many pencils does Alexa have now?

Step by step solution:
1. Alexa initially has 5 pencils.
2. She asks her father to bring 3 more pencils. Now she has 5 + 3 = 8 pencils.
3. She gives 2 pencils to her friend. Now she has 8 - 2 = 6 pencils.

Answer: Alexa hs 6 apples now.

Now solve the following logic step by step.

Problem: Michael bought 7 paintings. His friend Rafael is a fan of art and he asked Michael to gift him if possible. Michael gives Rafael 2 of the finest paintings he bought. Leonardo knowing that Michael is a fan of art gives him 4 brand new paintings of his. How many panitings does Michael have now?
[/INST]"""

Adding these demonstrations make it easier for the model to solve tasks.

**Disadvantage**
- We must have quality demonstration for several problems, and collecting such annotated data is expensive and time consuming.

**Advantages**
- Divides the task into manageable and understandable series of steps.

In [16]:
input_cot_prompt_tokens = tokenizer(cot_prompt, return_tensors = "pt")

In [22]:
output_cot_prompt = model.generate(**input_cot_prompt_tokens, max_new_tokens=500)
output_cot_prompt_text = tokenizer.decode(output_cot_prompt[0], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [23]:
output_cot_prompt_text

'[INST]\nLook at the following problem and how to solve it step by step.\nAlexa has 5 pencils. She asks her father to bring 3 more and then gives 2 to her friend. How many pencils does Alexa have now?\n\nStep by step solution:\n1. Alexa initially has 5 pencils.\n2. She asks her father to bring 3 more pencils. Now she has 5 + 3 = 8 pencils.\n3. She gives 2 pencils to her friend. Now she has 8 - 2 = 6 pencils.\n\nAnswer: Alexa hs 6 apples now.\n\nNow solve the following logic step by step.\n\nProblem: Michael bought 7 paintings. His friend Rafael is a fan of art and he asked Michael to gift him if possible. Michael gives Rafael 2 of the finest paintings he bought. Leonardo knowing that Michael is a fan of art gives him 4 brand new paintings of his. How many panitings does Michael have now?\n[/INST] Step by step solution:\n\n1. Michael initially bought 7 paintings.\n2. He gave 2 of the finest paintings to his friend Rafael. So now he has 7 - 2 = 5 paintings left.\n3. Leonardo gave him 4 b

**Zero shot CoT prompting**\
As we previously saw Mistral 7B is a model capable of handling zero shot prompting, now we see if the model can provide chain of thoughts when given a zero shot prompt.

In [24]:
zs_cot_prompt = """[INST]
Look at the following problem and how to solve it step by step.

Narayan released 7 songs and none of them were quite famous. Narayan died in 1795 and his songs felt relevant to the people in the 1800s. The single album with 7 songs was a great hit. Narayan had recorded 5 more albums, 2 with 7 songs in it and 3 with 5 songs. The music distributor released all of his songs in the 3 albums. How many songs does Narayan have released in total now?

[/INST]"""

In [25]:
input_zs_cot_prompt_tokens = tokenizer(zs_cot_prompt, return_tensors ="pt")

In [26]:
output_zs_cot_prompt = model.generate(**input_zs_cot_prompt_tokens, max_new_tokens=500)
output_zs_cot_prompt_text = tokenizer.decode(output_zs_cot_prompt[0], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [27]:
output_zs_cot_prompt_text

"[INST]\nLook at the following problem and how to solve it step by step.\n\nNarayan released 7 songs and none of them were quite famous. Narayan died in 1795 and his songs felt relevant to the people in the 1800s. The single album with 7 songs was a great hit. Narayan had recorded 5 more albums, 2 with 7 songs in it and 3 with 5 songs. The music distributor released all of his songs in the 3 albums. How many songs does Narayan have released in total now?\n\n[/INST] Let's break down the problem step by step:\n\n1. Narayan initially released 7 songs in a single album.\n2. He also recorded 5 more albums, with 2 albums containing 7 songs each and 3 albums containing 5 songs each.\n3. The music distributor released all of his songs in the 3 albums.\n\nNow let's calculate the total number of songs released by Narayan:\n\n1. From the initial single album, Narayan released 7 songs.\n2. From the 2 albums with 7 songs each, Narayan released 2 \\* 7 = 14 songs.\n3. From the 3 albums with 5 songs 

There are other techniques that can be used for prompt engineering and making the model adaptible to various tasks without fine tuning.

**Self Consistency**: The core idea behind Self Consistency is ensembling different models and bringing them to the right solution by majority of vote.
\
\
**Tree of Thought**: The model generates several reasoning intermediates and evaluates them by using search algorithms (Breadth First Search or Depth First Search)

These techniques allows greater reasoning capabilities but have higher computational cost since the model has to generate several responses and again select the most relevant answer.

**Declarative Self-Improving Language Programs in Python (DSPy)**

Above, we have manually created the prompts and it requires several hit and trials. DSPys try to standarize the prompting process and turns it into a kind of programming.

The authors of [DSPy](https://arxiv.org/abs/2310.03714) suggest that we can abstract prompts and fine tune them into signatures while prompting techniques are used as modules resulting **prompt engineering that are automated with optimizers.**

Given a dataset, we create a pipeline of DSPy containing signatures and modules, define which metric to optimize and then optimize. This process is then iterative; DSPy leads to optimizing prompts that we can use.

More on: \
[IBM DSPy](https://www.ibm.com/think/topics/dspy),\
[DSPy GitHub](https://github.com/stanfordnlp/dspy)